# Thick annular plates: shear deformation theories against exact and Ritz solutions

The annular-sector models of panels with transverse shear, `annular_fsdt_*` (first-order shear
deformation theory, FSDT) and `annular_tsdt_*` (Reddy's third-order theory, TSDT), are compared here
with three sources on thick isotropic annuli:

1. Bagheri, Kiani and Eslami [1]: buckling torques of moderately thick annuli, FSDT without a shear
   correction factor, by the differential quadrature method;
2. Bisadi, Es'haghi, Rokni and Ilkhani [2]: exact natural frequencies of annuli with Reddy's TSDT,
   for free, clamped, hard and soft simply supported edges;
3. Dozio [3]: Ritz natural frequencies of annuli with the FSDT and the shear correction factor
   $5/6$.

Section 1 shows, for [1], the buckling torques with the shear correction factor 1, which reproduces
the paper, beside those with the default of panels, `fsdt_shear_correction = 'rohwer'`.

In [1]:
import time

import numpy as np
from structsolve import freq

import panels
from panels.shell import Shell
from panels.multidomain import MultiDomain

print('panels', panels.__version__)
T_START = time.time()


def print_table(rows, headers):
    cols = list(zip(*([headers] + [[str(c) for c in r] for r in rows])))
    widths = [max(len(c) for c in col) for col in cols]
    fmt = '  '.join('%%%ds' % w for w in widths)
    print(fmt % tuple(headers))
    print('  '.join('-'*w for w in widths))
    for r in rows:
        print(fmt % tuple(str(c) for c in r))


def rel(value, ref):
    # relative difference in per cent
    return 100.*(value - ref)/ref

panels 0.15.0


## Model

A full annulus of outer radius $a$ and inner radius $b = R a$ is made of two half annuli whose free
radial edges are connected with `'SSycte'`, so that modes with any number $p$ of nodal diameters are
represented. In the coordinates of the annular sector, $x = a - \rho$ runs from the outer edge
($x = 0$, edge `x1`) inwards to the inner edge (edge `x2`), and $y$ is the arc length of the outer
edge; the rotations $\phi_x$, $\phi_y$ are the rotations $\psi_\rho$, $\psi_\theta$ of the sources up
to the sign of $\phi_x$. Each field is a Ritz series of Bardell's hierarchical functions, $m$ terms
along the radius and $n$ along each half circumference [4]. The material is isotropic, one layer,
$\nu = 0.3$; the results are dimensionless and do not depend on $E$, $\rho$ or $a$.

The edges, named inner-outer as in the sources, fix the following fields of panels:

| edge | TSDT [2] | FSDT [1], [3] |
|---|---|---|
| free (F) | none | none |
| clamped (C) | $u, v, w, w_{,x}, \phi_x, \phi_y$ | $u, v, w, \phi_x, \phi_y$ |
| hard simply supported ($S_h$, S) | $v, w, \phi_y$ | $v, w, \phi_y$ |
| soft simply supported ($S_s$) | $w$ | $w$ |

The TSDT has the slope $w_{,x}$ as a degree of freedom of the edge ([2], Eq. (50)); the FSDT does
not, and fixing $w_{,x}$ there would over-constrain a clamped edge. For the frequencies, the
in-plane conditions matter only for the in-plane modes: for one isotropic layer without offset, the
in-plane displacements $u, v$ are uncoupled from $w, \phi_x, \phi_y$.

The frequency parameter of [2] and [3] is $\beta = \omega a^2 \sqrt{\rho h/D}$, with
$D = E h^3/(12 (1 - \nu^2))$. The modes with $p \ge 1$ come in pairs of equal frequencies, merged
below; the count of each frequency (1 or 2) tells $p = 0$ from $p \ge 1$.

In [2]:
E, NU, RHO = 70e9, 0.3, 2700.
A = 1.
FIELDS = ('u', 'v', 'w', 'phix', 'phiy')
EDGES = {
    'tsdt': {'F': (), 'C': FIELDS + ('wr',), 'Sh': ('v', 'w', 'phiy'), 'Ss': ('w',)},
    'fsdt': {'F': (), 'C': FIELDS, 'Sh': ('v', 'w', 'phiy'), 'Ss': ('w',)},
    }
EDGES['fsdt']['S'] = EDGES['fsdt']['Sh']


def annulus(model, R, delta, inner, outer, m, n, shear=None, inplane=None):
    # full annulus of two half annuli; inplane fixes u, v on the given edges
    family = 'tsdt' if 'tsdt' in model else 'fsdt'
    halves = []
    for _ in range(2):
        s = Shell(model=model, a=A*(1 - R), b=np.pi*A, r=A, m=m, n=n,
                  plyt=delta*A, stack=[0], laminaprop=(E, NU), rho=RHO)
        if shear is not None:
            s.fsdt_shear_correction = shear
        for e in ('x1', 'x2', 'y1', 'y2'):
            for f in FIELDS:
                setattr(s, e + f, 1)
                setattr(s, e + f + 'r', 1)
        for e, kind in (('x1', outer), ('x2', inner)):
            for f in EDGES[family][kind]:
                setattr(s, e + f, 0)
        halves.append(s)
    p1, p2 = halves
    conn = [dict(p1=p1, p2=p2, func='SSycte', ycte1=p1.b, ycte2=0),
            dict(p1=p1, p2=p2, func='SSycte', ycte1=0, ycte2=p2.b)]
    return MultiDomain([p1, p2], conn)


def frequencies(md, delta, family='all', num=80):
    # beta of the modes of the family 'transverse' (w, phix, phiy dominate the Ritz
    # constants), 'inplane' or 'all', rigid-body modes excluded, pairs merged
    K = md.reduce(md.calc_kC(silent=True))
    M = md.reduce(md.calc_kM(silent=True))
    eigvals, eigvecs = freq(K, M, sparse_solver=False, silent=True, num_eigvalues=num,
                            symmetric=True, max_dense_size=10000)
    h = delta*A
    D = E*h**3/(12*(1 - NU**2))
    beta = np.sqrt(np.abs(eigvals))*A**2*np.sqrt(RHO*h/D)
    phi = md.expand(eigvecs)
    transverse = np.zeros(phi.shape[0], dtype=bool)
    for p in md.panels:
        for field in (2, 3, 4):
            transverse[p.col_start + field:p.col_end:5] = True
    frac = np.linalg.norm(phi[transverse], axis=0)/np.linalg.norm(phi, axis=0)
    keep = np.isfinite(beta) & (beta > 0.1)
    if family == 'transverse':
        keep &= frac > 0.5
    elif family == 'inplane':
        keep &= frac < 0.5
    beta = np.sort(beta[keep])
    values, count = [], []
    for b in beta:
        if values and b/values[-1] - 1 < 1e-4:
            count[-1] += 1
        else:
            values.append(b)
            count.append(1)
    return np.array(values), np.array(count)

## 1. Buckling torque of moderately thick annuli (Bagheri, Kiani and Eslami 2019)

A torque $T$ on the outer edge, reacted on the inner edge, gives the membrane state
$N_{\rho\theta} = T/(2 \pi \rho^2)$, $N_{\rho\rho} = N_{\theta\theta} = 0$ ([1], Eqs. (10) and (12)).
It is applied as the tangential force per unit length $q = T/(2 \pi a^2)$ on the outer edge, the
inner edge fixed in its plane ($u = v = 0$); `MultiDomain.linear_buckling` computes the linear
static state and the least buckling load. The critical torque is $\tau = T/(2 \pi D)$ ([1], Eq.
(26)). Here the simply supported edge (S) of [1] is the hard one, $w = \phi_y = 0$ ([1], Eq. (11)).

[1] uses the transverse shear stiffness $A_{44} = A_{55} = G h$ without a shear correction factor
(its Eq. (7)): `fsdt_shear_correction = 1` reproduces its Table 2. The default of panels,
`'rohwer'`, the method of Rohwer for the transverse shear stiffness of laminates, gives the factor
printed below for the homogeneous plate:

In [3]:
s = Shell(model='annular_fsdt_donnell', a=0.7, b=np.pi, r=1., m=4, n=4, plyt=0.1, stack=[0],
          laminaprop=(E, NU), rho=RHO)
G = E/(2*(1 + NU))
print('transverse shear stiffness / (G h), rohwer:', np.diag(s.ABD)[6:]/(G*0.1))

transverse shear stiffness / (G h), rohwer: [0.83333333 0.83333333]


In [4]:
def tau(beta, inner, outer, delta, shear, m=12, n=16):
    D = E*(delta*A)**3/(12*(1 - NU**2))
    T = 2*np.pi*D
    flags = {'C': (0, 0, 0), 'S': (0, 1, 0), 'F': (1, 1, 1)}   # w, phix, phiy
    halves = []
    for _ in range(2):
        s = Shell(model='annular_fsdt_donnell', a=A*(1 - beta), b=np.pi*A, r=A, m=m, n=n,
                  plyt=delta*A, stack=[0], laminaprop=(E, NU), rho=RHO)
        s.fsdt_shear_correction = shear
        for e in ('x1', 'x2', 'y1', 'y2'):
            for f in FIELDS:
                setattr(s, e + f, 1)
                setattr(s, e + f + 'r', 1)
        s.x1w, s.x1phix, s.x1phiy = flags[outer]
        s.x2w, s.x2phix, s.x2phiy = flags[inner]
        s.x2u = s.x2v = 0
        s.add_distr_load_fixed_x(0., funcy=lambda y: T/(2*np.pi*A**2) + 0*y)
        halves.append(s)
    p1, p2 = halves
    conn = [dict(p1=p1, p2=p2, func='SSycte', ycte1=p1.b, ycte2=0),
            dict(p1=p1, p2=p2, func='SSycte', ycte1=0, ycte2=p2.b)]
    eigvals, _ = MultiDomain([p1, p2], conn).linear_buckling(num_eigvalues=6, silent=True)
    return np.abs(eigvals).min()


print('convergence, C-C, h/a = 0.10, beta = 0.5, shear correction 1')
rows = []
for m, n in [(8, 12), (10, 14), (12, 16), (14, 18)]:
    rows.append([m, n, '%.4f' % tau(0.5, 'C', 'C', 0.10, 1.)])
print_table(rows, ['m', 'n', 'tau'])

convergence, C-C, h/a = 0.10, beta = 0.5, shear correction 1


 m   n      tau
--  --  -------
 8  12  98.4361
10  14  98.4361
12  16  98.4361
14  18  98.4361


Table 2 of [1], edges inner-outer, $\beta = b/a$ (here the ratio of the radii): `k = 1` is panels
with the shear correction factor 1, as in [1]; `rohwer` with the default of panels. Each column has
its difference from [1]; the last column is the effect of the shear correction factor alone.

In [5]:
BAGHERI = {   # [1] Table 2, tau for beta = 0.3 and 0.5
    (('C', 'C'), 0.05): (53.758, 150.414), (('C', 'C'), 0.10): (39.261, 98.436),
    (('S', 'S'), 0.05): (34.976, 100.749), (('S', 'S'), 0.10): (29.912, 79.632),
    (('C', 'F'), 0.05): (12.215, 33.524), (('C', 'F'), 0.10): (11.030, 29.174),
    (('F', 'C'), 0.05): (9.904, 29.607), (('F', 'C'), 0.10): (8.877, 25.626),
    }
rows = []
for (edges, delta), refs in BAGHERI.items():
    for beta, ref in zip((0.3, 0.5), refs):
        t1 = tau(beta, *edges, delta, 1.)
        tr = tau(beta, *edges, delta, 'rohwer')
        rows.append(['-'.join(edges), '%.2f' % delta, beta, '%.3f' % ref, '%.3f' % t1,
                     '%+.3f' % rel(t1, ref), '%.3f' % tr, '%+.2f' % rel(tr, ref),
                     '%+.2f' % rel(tr, t1)])
print_table(rows, ['edges', 'h/a', 'beta', 'tau [1]', 'k = 1', 'diff (%)', 'rohwer',
                   'diff (%)', 'rohwer vs k = 1 (%)'])

edges   h/a  beta  tau [1]    k = 1  diff (%)   rohwer  diff (%)  rohwer vs k = 1 (%)
-----  ----  ----  -------  -------  --------  -------  --------  -------------------
  C-C  0.05   0.3   53.758   53.760    +0.005   52.421     -2.49                -2.49
  C-C  0.05   0.5  150.414  150.415    +0.001  145.088     -3.54                -3.54
  C-C  0.10   0.3   39.261   39.262    +0.002   36.699     -6.53                -6.53
  C-C  0.10   0.5   98.436   98.436    +0.000   90.368     -8.20                -8.20
  S-S  0.05   0.3   34.976   34.976    -0.000   34.584     -1.12                -1.12
  S-S  0.05   0.5  100.749  100.749    +0.000   98.990     -1.75                -1.75
  S-S  0.10   0.3   29.912   29.912    +0.001   28.804     -3.70                -3.71
  S-S  0.10   0.5   79.632   79.632    +0.000   75.433     -5.27                -5.27
  C-F  0.05   0.3   12.215   12.216    +0.008   12.122     -0.76                -0.77
  C-F  0.05   0.5   33.524   33.524    +0.000   33.165

## 2. Natural frequencies with Reddy's TSDT (Bisadi et al. 2012)

[2] solves exactly the equations of motion of Reddy's TSDT for the transverse vibration of an
isotropic annulus, with the rotary and higher-order inertia, and tabulates $\beta$ by $(p, s)$, $p$
the number of nodal diameters and $s$ the radial order, for $R = b/a$ and $\delta = h/a$. The model
`annular_tsdt_donnell` has the same displacement field (Reddy's, about the mid-surface) and its
consistent mass matrix; only the transverse family of modes is compared.

**Convergence.** The clamped edges converge most slowly, as the TSDT clamps the slope $w_{,x}$ in
addition to the rotations; the slowest is the clamped inner edge of radius $0.1 a$:

In [6]:
MESHES = [(10, 12), (14, 16), (16, 20), (18, 22)]
for case in [('C', 'C', 0.1, 0.1), ('C', 'C', 0.2, 0.2), ('Sh', 'Sh', 0.2, 0.2)]:
    inner, outer, R, delta = case
    rows = []
    for m, n in MESHES:
        v, c = frequencies(annulus('annular_tsdt_donnell', R, delta, inner, outer, m, n),
                           delta, 'transverse')
        rows.append([m, n] + ['%.4f' % b for b in v[:6]])
    print('%s-%s, R = %.1f, h/a = %.1f' % case)
    print_table(rows, ['m', 'n'] + ['beta_%d' % (i + 1) for i in range(6)])
    print()
M_FIN, N_FIN = 16, 20

C-C, R = 0.1, h/a = 0.1
 m   n   beta_1   beta_2   beta_3   beta_4   beta_5   beta_6
--  --  -------  -------  -------  -------  -------  -------
10  12  24.8565  26.1706  33.3278  46.0369  60.9508  63.1101
14  16  24.7639  26.0566  33.2452  45.9798  60.8685  62.7539
16  20  24.7484  26.0383  33.2324  45.9703  60.8547  62.6947
18  22  24.7406  26.0292  33.2262  45.9658  60.8481  62.6657



C-C, R = 0.2, h/a = 0.2
 m   n   beta_1   beta_2   beta_3   beta_4   beta_5   beta_6
--  --  -------  -------  -------  -------  -------  -------
10  12  24.9452  25.9055  30.0598  37.8795  47.7121  55.6101
14  16  24.8281  25.7896  29.9594  37.7972  47.6284  55.3435
16  20  24.8195  25.7812  29.9523  37.7916  47.6230  55.3263
18  22  24.8175  25.7792  29.9506  37.7903  47.6218  55.3225



Sh-Sh, R = 0.2, h/a = 0.2
 m   n   beta_1   beta_2   beta_3   beta_4   beta_5   beta_6
--  --  -------  -------  -------  -------  -------  -------
10  12  14.7124  16.7693  23.0985  32.3675  42.8742  46.0566
14  16  14.7123  16.7693  23.0984  32.3674  42.8740  46.0564
16  20  14.7123  16.7693  23.0984  32.3674  42.8740  46.0564
18  22  14.7123  16.7693  23.0984  32.3674  42.8740  46.0564



**Axisymmetric modes, Table 5 of [2].** $R = 0.1$, $\delta = 0.1$, all nine combinations of free,
hard simply supported and clamped edges; $(0, 0)$ and $(0, 1)$ are the two lowest single
frequencies of the transverse family. [2] lists beside them the exact FSDT values of Irie, Yamada
and Takagi [5], which do not include the shear correction factor used; panels with
`annular_fsdt_donnell` is shown for the factors $5/6$ and $\pi^2/12$ as a sensitivity (with $m = 14$,
$n = 16$: without the slope $w_{,x}$ at a clamped edge the FSDT converges faster, see Section 3).

In [7]:
BISADI_T5 = {   # [2] Table 5, (inner, outer): (0,0) and (0,1), HSDT and FSDT [5]
    ('C', 'C'): ((24.7343, 62.6432), (24.63, 62.14)),
    ('C', 'Sh'): ((16.6235, 52.3407), (16.57, 52.06)),
    ('C', 'F'): ((4.15336, 23.2065), (4.15, 23.12)),
    ('Sh', 'C'): ((21.2294, 57.1018), (21.20, 56.91)),
    ('Sh', 'Sh'): ((13.8817, 46.9999), (13.87, 46.95)),
    ('Sh', 'F'): ((3.40091, 19.7975), (3.40, 19.79)),
    ('F', 'C'): ((9.90133, 36.3963), (9.90, 36.33)),
    ('F', 'Sh'): ((4.81445, 28.0500), (4.81, 28.04)),
    ('F', 'F'): ((8.64765, 35.9696), (8.65, 35.95)),
    }
rows = []
for (inner, outer), (hsdt, fsdt) in BISADI_T5.items():
    v, c = frequencies(annulus('annular_tsdt_donnell', 0.1, 0.1, inner, outer, M_FIN, N_FIN),
                       0.1, 'transverse')
    tsdt = v[c == 1][:2]
    f = {}
    for k in (5/6, np.pi**2/12):
        v, c = frequencies(annulus('annular_fsdt_donnell', 0.1, 0.1, inner, outer, 14, 16,
                                   shear=k), 0.1, 'transverse')
        f[k] = v[c == 1][:2]
    for i, mode in enumerate(('(0,0)', '(0,1)')):
        rows.append(['%s-%s' % (inner, outer), mode, hsdt[i], '%.4f' % tsdt[i],
                     '%+.3f' % rel(tsdt[i], hsdt[i]), fsdt[i],
                     '%.3f' % f[5/6][i], '%+.2f' % rel(f[5/6][i], fsdt[i]),
                     '%.3f' % f[np.pi**2/12][i], '%+.2f' % rel(f[np.pi**2/12][i], fsdt[i])])
print_table(rows, ['edges', '(p,s)', 'TSDT [2]', 'panels TSDT', 'diff (%)', 'FSDT [5]',
                   'k = 5/6', 'diff (%)', 'k = pi^2/12', 'diff (%)'])

edges  (p,s)  TSDT [2]  panels TSDT  diff (%)  FSDT [5]  k = 5/6  diff (%)  k = pi^2/12  diff (%)
-----  -----  --------  -----------  --------  --------  -------  --------  -----------  --------
  C-C  (0,0)   24.7343      24.7484    +0.057     24.63   24.657     +0.11       24.629     -0.00
  C-C  (0,1)   62.6432      62.6947    +0.082     62.14   62.257     +0.19       62.140     -0.00
 C-Sh  (0,0)   16.6235      16.6300    +0.039     16.57   16.588     +0.11       16.575     +0.03
 C-Sh  (0,1)   52.3407      52.3782    +0.072     52.06   52.130     +0.13       52.056     -0.01
  C-F  (0,0)   4.15336       4.1538    +0.010      4.15    4.151     +0.02        4.150     -0.00
  C-F  (0,1)   23.2065      23.2177    +0.048     23.12   23.145     +0.11       23.124     +0.02
 Sh-C  (0,0)   21.2294      21.2305    +0.005      21.2   21.216     +0.08       21.200     -0.00
 Sh-C  (0,1)   57.1018      57.1110    +0.016     56.91   56.993     +0.15       56.912     +0.00
Sh-Sh  (0,0)   13.88

**The eight lowest frequencies, Tables 6 to 9 of [2].** Selected thick annuli with every kind of
edge; the count of each frequency is 1 for $p = 0$ and 2 otherwise:

In [8]:
BISADI_EIGHT = {   # [2] Tables 6-9, (inner, outer, R, delta): beta and (p, s)
    ('C', 'C', 0.2, 0.2): ((24.817, 25.7787, 29.9502, 37.79, 47.6215, 55.3217, 56.8944, 58.1922),
                           ('0,0', '1,0', '2,0', '3,0', '4,0', '0,1', '1,1', '5,0')),
    ('C', 'F', 0.2, 0.2): ((4.22136, 4.82685, 5.76949, 11.3957, 18.8497, 24.1187, 25.8523, 27.3118),
                           ('1,0', '0,0', '2,0', '3,0', '4,0', '0,1', '1,1', '5,0')),
    ('Ss', 'Ss', 0.4, 0.1): ((26.7123, 28.1832, 32.9288, 41.3253, 53.0778, 67.4162, 83.5587,
                              93.7523),
                             ('0,0', '1,0', '2,0', '3,0', '4,0', '5,0', '6,0', '0,1')),
    ('Sh', 'Sh', 0.2, 0.2): ((14.7123, 16.7693, 23.0984, 32.3674, 42.874, 46.0564, 48.2819,
                              53.8528),
                             ('0,0', '1,0', '2,0', '3,0', '4,0', '0,1', '1,1', '5,0')),
    ('F', 'F', 0.2, 0.3): ((4.62924, 7.55348, 10.3689, 14.6343, 16.6936, 23.417, 23.7821, 28.8167),
                           ('2,0', '0,0', '3,0', '1,0', '4,0', '5,0', '2,1', '0,1')),
    ('F', 'Sh', 0.4, 0.2): ((4.62418, 10.3882, 19.6467, 30.1333, 37.5164, 40.1593, 41.4177,
                             47.4932),
                            ('0,0', '1,0', '2,0', '3,0', '0,1', '1,1', '4,0', '2,1')),
    }
for (inner, outer, R, delta), (ref, modes) in BISADI_EIGHT.items():
    v, c = frequencies(annulus('annular_tsdt_donnell', R, delta, inner, outer, M_FIN, N_FIN),
                       delta, 'transverse')
    rows = [[modes[i], ref[i], '%.4f' % v[i], '%+.4f' % rel(v[i], ref[i]), c[i]]
            for i in range(8)]
    print('%s-%s, R = %.1f, h/a = %.1f' % (inner, outer, R, delta))
    print_table(rows, ['(p,s) [2]', 'beta [2]', 'panels', 'diff (%)', 'count'])
    print()

C-C, R = 0.2, h/a = 0.2
(p,s) [2]  beta [2]   panels  diff (%)  count
---------  --------  -------  --------  -----
      0,0    24.817  24.8195   +0.0101      1
      1,0   25.7787  25.7812   +0.0096      2
      2,0   29.9502  29.9523   +0.0069      2
      3,0     37.79  37.7916   +0.0042      2
      4,0   47.6215  47.6230   +0.0031      2
      0,1   55.3217  55.3263   +0.0083      1
      1,1   56.8944  56.8987   +0.0075      2
      5,0   58.1922  58.1938   +0.0028      2



C-F, R = 0.2, h/a = 0.2
(p,s) [2]  beta [2]   panels  diff (%)  count
---------  --------  -------  --------  -----
      1,0   4.22136   4.2215   +0.0038      2
      0,0   4.82685   4.8269   +0.0018      1
      2,0   5.76949   5.7696   +0.0022      2
      3,0   11.3957  11.3957   -0.0002      2
      4,0   18.8497  18.8497   +0.0002      2
      0,1   24.1187  24.1205   +0.0075      1
      1,1   25.8523  25.8539   +0.0063      2
      5,0   27.3118  27.3118   -0.0001      2



Ss-Ss, R = 0.4, h/a = 0.1
(p,s) [2]  beta [2]   panels  diff (%)  count
---------  --------  -------  --------  -----
      0,0   26.7123  26.7123   +0.0002      1
      1,0   28.1832  28.1832   -0.0000      2
      2,0   32.9288  32.9288   +0.0001      2
      3,0   41.3253  41.3253   +0.0001      2
      4,0   53.0778  53.0778   +0.0000      2
      5,0   67.4162  67.4162   -0.0000      2
      6,0   83.5587  83.5587   -0.0000      2
      0,1   93.7523  93.7523   +0.0000      1



Sh-Sh, R = 0.2, h/a = 0.2
(p,s) [2]  beta [2]   panels  diff (%)  count
---------  --------  -------  --------  -----
      0,0   14.7123  14.7123   +0.0002      1
      1,0   16.7693  16.7693   -0.0001      2
      2,0   23.0984  23.0984   -0.0000      2
      3,0   32.3674  32.3674   +0.0001      2
      4,0    42.874  42.8740   -0.0001      2
      0,1   46.0564  46.0564   +0.0001      1
      1,1   48.2819  48.2819   -0.0001      2
      5,0   53.8528  53.8528   -0.0001      2



F-F, R = 0.2, h/a = 0.3
(p,s) [2]  beta [2]   panels  diff (%)  count
---------  --------  -------  --------  -----
      2,0   4.62924   4.6292   +0.0001      2
      0,0   7.55348   7.5535   +0.0000      1
      3,0   10.3689  10.3689   -0.0003      2
      1,0   14.6343  14.6343   -0.0002      2
      4,0   16.6936  16.6936   +0.0000      2
      5,0    23.417  23.4170   -0.0000      2
      2,1   23.7821  23.7821   -0.0001      2
      0,1   28.8167  28.8168   +0.0002      1



F-Sh, R = 0.4, h/a = 0.2
(p,s) [2]  beta [2]   panels  diff (%)  count
---------  --------  -------  --------  -----
      0,0   4.62418   4.6242   +0.0000      1
      1,0   10.3882  10.3882   -0.0003      2
      2,0   19.6467  19.6467   -0.0002      2
      3,0   30.1333  30.1333   +0.0001      2
      0,1   37.5164  37.5164   +0.0000      1
      1,1   40.1593  40.1593   -0.0001      2
      4,0   41.4177  41.4177   -0.0001      2
      2,1   47.4932  47.4932   -0.0000      2



## 3. Natural frequencies with the FSDT (Dozio 2015)

[3], Table 5, gives the eight lowest $\beta$ of annuli with $R = 0.3$ and $h/a = 0.2$ for a
first-order theory obtained from its HOT$_1$ model with the transverse normal stress set to zero and
the shear correction factor $5/6$ (its Eq. (30)), for the edges free-hard simply supported (FS),
free-free (FF), free-clamped (FC) and clamped-clamped (CC); its simply supported edge fixes $v$ and $w$
and the rotation $\phi_y$, its clamped edge all the displacements ([3], Table 1). The model is
`annular_fsdt_donnell` with `fsdt_shear_correction = 5/6`. [3] includes the in-plane torsional mode,
$p = 0$; the table below lists the modes of both families of panels, the transverse family marked
`T`, the in-plane one `I`:

In [9]:
DOZIO = {   # [3] Table 5, (inner, outer)
    ('F', 'S'): (4.5328, 11.205, 12.711, 15.875, 20.740, 27.931, 30.530, 31.301),
    ('F', 'F'): (4.6160, 7.8878, 11.126, 15.146, 15.659, 18.778, 26.674, 27.279),
    ('F', 'C'): (10.365, 15.895, 25.384, 35.763, 36.860, 39.424, 40.534, 43.901),
    ('C', 'C'): (30.146, 30.878, 33.775, 39.638, 47.999, 48.220, 52.997, 57.793),
    }
for (inner, outer), ref in DOZIO.items():
    md_ = annulus('annular_fsdt_donnell', 0.3, 0.2, inner, outer, M_FIN, N_FIN, shear=5/6)
    vt, ct = frequencies(md_, 0.2, 'transverse')
    vi, ci = frequencies(md_, 0.2, 'inplane')
    allm = sorted([(b, 'T', k) for b, k in zip(vt, ct)] + [(b, 'I', k) for b, k in zip(vi, ci)])
    rows = []
    j = 0
    for b, fam, k in allm:
        if j >= len(ref) or b > 1.02*ref[-1]:
            break
        if abs(b/ref[j] - 1) < 2e-3:
            rows.append([j + 1, ref[j], '%.4f' % b, '%+.4f' % rel(b, ref[j]), fam, k])
            j += 1
        else:
            rows.append(['', '', '%.4f' % b, 'not in [3]', fam, k])
    print('%s%s' % (inner, outer))
    print_table(rows, ['mode [3]', 'beta [3]', 'panels', 'diff (%)', 'family', 'count'])
    print()

FS
mode [3]  beta [3]   panels  diff (%)  family  count
--------  --------  -------  --------  ------  -----
       1    4.5328   4.5328   +0.0000       T      1
       2    11.205  11.2051   +0.0013       T      2
       3    12.711  12.7107   -0.0021       I      2
       4    15.875  15.8746   -0.0027       I      2
       5     20.74  20.7402   +0.0010       T      2
       6    27.931  27.9309   -0.0002       I      1
       7     30.53  30.5300   -0.0001       T      1
       8    31.301  31.3012   +0.0007       T      2



FF
mode [3]  beta [3]   panels  diff (%)  family  count
--------  --------  -------  --------  ------  -----
       1     4.616   4.6160   +0.0003       T      2
       2    7.8878   7.8878   -0.0002       T      1
       3    11.126  11.1257   -0.0027       T      2
       4    15.146  15.1458   -0.0011       T      2
       5    15.659  15.6594   +0.0023       I      2
       6    18.778  18.7783   +0.0015       T      2
       7    26.674  26.6744   +0.0015       T      2
       8    27.279  27.2790   +0.0001       T      2



FC
mode [3]  beta [3]   panels    diff (%)  family  count
--------  --------  -------  ----------  ------  -----
       1    10.365  10.3645     -0.0045       T      1
       2    15.895  15.8954     +0.0023       T      2
       3    25.384  25.3842     +0.0008       T      2
       4    35.763  35.7631     +0.0004       T      2
       5     36.86  36.8597     -0.0007       T      1
       6    39.424  39.4235     -0.0012       I      2
                    40.2940  not in [3]       T      2
       7    40.534  40.5341     +0.0002       I      1
       8    43.901  43.9008     -0.0004       I      2



CC
mode [3]  beta [3]   panels  diff (%)  family  count
--------  --------  -------  --------  ------  -----
       1    30.146  30.1458   -0.0005       T      1
       2    30.878  30.8782   +0.0007       T      2
       3    33.775  33.7745   -0.0015       T      2
       4    39.638  39.6378   -0.0004       T      2
       5    47.999  47.9986   -0.0008       T      2
       6     48.22  48.2199   -0.0003       I      1
       7    52.997  52.9966   -0.0007       I      2
       8    57.793  57.7927   -0.0005       T      2



## 4. Discussion

**Bagheri et al. [1].** With the shear correction factor 1 of [1], panels reproduces the 16 buckling
torques of its Table 2 within 0.04 % (0.01 % except F-C, $h/a = 0.05$, $\beta = 0.3$), converged
for $m = 8$, $n = 12$ in the case shown. The default of panels, `'rohwer'`, gives the factor $5/6$ for the
homogeneous plate, and torques lower by 0.8 % ($h/a = 0.05$, edges C-F and F-C) to 8.2 % ($h/a =
0.10$, C-C, $\beta = 0.5$): the more the transverse shear lowers the torque, the thicker and the more
restrained the plate, the larger the effect of the factor. The values of [1] are those of a plate
without a shear correction, stiffer in transverse shear than the homogeneous plate.

**Bisadi et al. [2], TSDT.** The Ritz solution approaches the exact one from above. Without a
clamped edge, panels reproduces the tabulated values to their last digit (within 0.0003 %) with
$m = 16$, $n = 20$, the modes in the order of [2] and with the count expected from $p$. A clamped
edge, where the TSDT fixes $w_{,x}$ besides the rotations, converges more slowly: up to 0.01 %
for $R = 0.2$, and 0.06 % to 0.08 % for the clamped inner edge of radius $0.1 a$ (Table 5), still
decreasing with $m$ and $n$. The thin annuli of the tests of `annular_tsdt_*` reproduce the classical
plate; these tables check the thick range, $h/a$ up to 0.3, for all four kinds of edge.

**Irie et al. [5], FSDT.** [2] reprints the exact FSDT values of [5] without its shear correction
factor. `annular_fsdt_donnell` with $\pi^2/12$ reproduces them within 0.04 % (0.09 % for F-$S_h$,
$(0, 0)$, 4.81 in [5]), whereas $5/6$ gives up to 0.19 % more: the values of [5] are consistent with
$\pi^2/12$. The TSDT frequencies are above the FSDT ones by up to 0.8 % here, as
[2] notes.

**Dozio [3], FSDT.** With the shear correction $5/6$, panels reproduces the 32 tabulated values within
0.005 %, transverse and in-plane modes alike, including the in-plane torsional mode $p = 0$ of the
annuli with a clamped edge. One mode of panels is not in Table 5 of [3]: for the annulus free
inside and clamped outside, a transverse pair at $\beta = 40.294$, one nodal diameter, second radial
order, between the sixth and the seventh values of [3]; `annular_tsdt_donnell` has the same mode at
40.720. The lists of the 3-D Ritz solutions reprinted in [3], Table 4, for the same annulus do not
have it either; the reason is not known.

## References

1. Bagheri, H., Kiani, Y., and Eslami, M. R., "Buckling of moderately thick annular plates subjected
   to torque", Archive of Mechanical Engineering, Vol. 66, No. 2, 2019, pp. 209-227, Eqs. (7), (10)-(12)
   and (26), Table 2. https://doi.org/10.24425/ame.2019.128445
2. Bisadi, H., Es'haghi, M., Rokni, H., and Ilkhani, M., "Benchmark solution for transverse vibration
   of annular Reddy plates", International Journal of Mechanical Sciences, Vol. 56, 2012, pp. 35-49,
   Eqs. (47)-(50), Tables 5-9. https://doi.org/10.1016/j.ijmecsci.2011.12.007
3. Dozio, L., "Refined 2-D theories for free vibration analysis of annular plates: Unified Ritz
   formulation and numerical assessment", Computers and Structures, Vol. 147, 2015, pp. 250-258,
   Eq. (30), Tables 1 and 5. https://doi.org/10.1016/j.compstruc.2014.10.015
4. Castro, S. G. P., and Donadon, M. V., "Assembly of semi-analytical models to address linear
   buckling and vibration of stiffened composite panels with debonding defect", Composite
   Structures, Vol. 160, 2017, pp. 232-247. https://doi.org/10.1016/j.compstruct.2016.10.026
5. Irie, T., Yamada, G., and Takagi, K., "Natural frequencies of thick annular plates", Journal of
   Applied Mechanics, Vol. 49, 1982, pp. 633-638, as reprinted in [2], Table 5.
   https://doi.org/10.1115/1.3162539

In [10]:
print('total run time: %.1f min' % ((time.time() - T_START)/60))

total run time: 47.5 min
